In [ ]:
# Setup for Colab / notebook use
# Run this once. It is rerun-safe: existing folders/files are reused.
!pip install rdkit pandas numpy scikit-learn scipy -q

import os

if not os.path.isdir("scscore"):
    !git clone https://github.com/connorcoley/scscore.git
else:
    print("scscore folder already exists; skipping clone")

if not os.path.exists("sascorer.py"):
    !wget -q https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py
if not os.path.exists("fpscores.pkl.gz"):
    !wget -q https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz

print("Setup complete")


In [ ]:

import os
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, FilterCatalog, QED, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")   # suppress kekulization error noise
RDLogger.DisableLog("rdApp.info")    # suppress "Running X" spam

try:
    from rdkit.Chem import rdFingerprintGenerator
    _MORGAN_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
except Exception:
    _MORGAN_GENERATOR = None


def morgan_fp(mol):
    if _MORGAN_GENERATOR is not None:
        return _MORGAN_GENERATOR.GetFingerprint(mol)
    from rdkit.Chem import AllChem
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)


# -----------------------------
# CONFIG
# -----------------------------
GENERATED_FILE = "high_activity_candidates.csv"
REFERENCE_FILE = "Active_Smiles.csv"
GENERATED_SMILES_COL = "SMILES"
REFERENCE_SMILES_COL = "canonical_smiles"

OUTPUT_PREFIX = "AKT1_small_pool_curated"

# Near-dedup: fine to always run at this scale (a few thousand items).
NEAR_DUP_THRESH = 0.95
ENABLE_NEAR_DEDUP = True

# Novelty: avoid exact training-set repeats, but keep molecules that are still AKT1-like.
SIM_NOVEL_UPPER = 0.90
SIM_RELEVANCE_LOWER = 0.15

# Relaxed hard filters. These are meant to remove obvious bad cases, not over-curate.
MIN_MW = 100
MAX_MW = 800
MAX_RO5_VIOLATIONS = 2
MAX_TPSA = 200
MAX_ROTB = 18
MIN_QED = 0.20
MIN_LOGS = -9.0
MAX_ABS_CHARGE = 2

# Alerts: annotate most alerts, hard-filter only strongly reactive motifs by default.
FILTER_PAINS = False
FILTER_BRENK = False
FILTER_NIH = False
FILTER_WARHEADS = True
FILTER_AGGREGATORS = False


# -----------------------------
# Load data
# -----------------------------
print("Loading data ...")
generated_df = pd.read_csv(GENERATED_FILE)
training_df = pd.read_csv(REFERENCE_FILE)

generated_smiles = generated_df[GENERATED_SMILES_COL].dropna().astype(str).tolist()
reference_smiles = training_df[REFERENCE_SMILES_COL].dropna().astype(str).tolist()

funnel = {"Input": len(generated_smiles)}
print(f"  Generated SMILES: {len(generated_smiles):,}")
print(f"  Reference SMILES: {len(reference_smiles):,}")


# -----------------------------
# Standardization  (Option 1: robust kekulization recovery)
# -----------------------------
_largest_frag = rdMolStandardize.LargestFragmentChooser()
_normalizer = rdMolStandardize.Normalizer()
_uncharger = rdMolStandardize.Uncharger()
_tautomer_enum = rdMolStandardize.TautomerEnumerator()

ALLOWED_ATOMS = {1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53}  # H/B/C/N/O/F/Si/P/S/Cl/Br/I


def _parse_smiles_robust(smiles):
    """
    Try normal RDKit parsing first. If kekulization fails, retry with
    sanitize=False, clear aromatic flags, and re-sanitize manually.
    Returns mol or None.
    """
    # Attempt 1: standard parse
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return mol

    # Attempt 2: parse without sanitization, fix aromaticity, then sanitize
    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        return None

    # Clear all aromaticity flags — let RDKit re-determine them
    for atom in mol.GetAtoms():
        atom.SetIsAromatic(False)
    for bond in mol.GetBonds():
        bond.SetIsAromatic(False)
        # Reset bond order to single if it was aromatic (RDKit uses bond type 1.5)
        if bond.GetBondTypeAsDouble() == 1.5:
            bond.SetBondType(Chem.BondType.SINGLE)

    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None


def standardize(smiles):
    try:
        mol = _parse_smiles_robust(smiles)
        if mol is None:
            return None, None

        mol = _largest_frag.choose(mol)
        mol = _normalizer.normalize(mol)
        mol = _uncharger.uncharge(mol)
        mol = _tautomer_enum.Canonicalize(mol)
        Chem.SanitizeMol(mol)

        if any(atom.GetAtomicNum() not in ALLOWED_ATOMS for atom in mol.GetAtoms()):
            return None, None

        mw = Descriptors.MolWt(mol)
        if mw < MIN_MW or mw > MAX_MW:
            return None, None

        return Chem.MolToSmiles(mol, canonical=True), mol
    except Exception:
        return None, None


print("Standardizing molecules ...")
standardized = []
for smi in generated_smiles:
    can, mol = standardize(smi)
    if can is not None:
        standardized.append((can, mol))

funnel["Standardized"] = len(standardized)
print(f"  Standardized: {len(standardized):,}")


# -----------------------------
# Exact and near deduplication
# -----------------------------
print("Deduplicating ...")
seen = {}
for smi, mol in standardized:
    if smi not in seen:
        seen[smi] = mol
exact_unique_items = list(seen.items())
funnel["Exact unique"] = len(exact_unique_items)
print(f"  Exact unique: {len(exact_unique_items):,}")


def loose_near_dedup(items, threshold=0.98):
    """Greedy near-dedup. Fine to run on a few thousand items."""
    fps = [morgan_fp(mol) for _, mol in items]
    kept_indices = []
    dropped = set()
    for i, fp_i in enumerate(fps):
        if i in dropped:
            continue
        kept_indices.append(i)
        remaining_idx = [j for j in range(i + 1, len(fps)) if j not in dropped]
        if not remaining_idx:
            break
        sims = DataStructs.BulkTanimotoSimilarity(fp_i, [fps[j] for j in remaining_idx])
        for j, sim in zip(remaining_idx, sims):
            if sim >= threshold:
                dropped.add(j)
    return [items[i] for i in kept_indices]


if ENABLE_NEAR_DEDUP:
    unique_items = loose_near_dedup(exact_unique_items, NEAR_DUP_THRESH)
    print(f"  Near-dedup at Tanimoto >= {NEAR_DUP_THRESH}: {len(unique_items):,}")
else:
    unique_items = exact_unique_items

funnel["Deduplicated"] = len(unique_items)


# -----------------------------
# Alerts: annotate first, hard-filter only selected classes
# -----------------------------
def make_catalog(*catalog_names):
    params = FilterCatalog.FilterCatalogParams()
    for name in catalog_names:
        params.AddCatalog(name)
    return FilterCatalog.FilterCatalog(params)

FC = FilterCatalog.FilterCatalogParams.FilterCatalogs
pains_catalog = make_catalog(FC.PAINS)
brenk_catalog = make_catalog(FC.BRENK)
nih_catalog = make_catalog(FC.NIH)

WARHEAD_SMARTS = [
    "[CX3](=[OX1])[F,Cl,Br,I]",       # acid halide
    "[NX3][CX3](=[OX1])[CX3]=[CX3]",  # acrylamide-like
    "[CX3]=[CX3]C#N",                 # acrylonitrile
    "[C;!$(C=*)][Cl,Br,I]",           # alkyl halide (aromatic C-halide is NOT flagged)
    "[N+]#[N-]",                       # diazonium/azide-like ionic N patterns
]
AGGREGATOR_SMARTS = [
    "c1ccc(cc1)C(=O)c1ccccc1",
    "c1ccc(cc1)/C=C/C(=O)",
]
warhead_mols = [Chem.MolFromSmarts(s) for s in WARHEAD_SMARTS if Chem.MolFromSmarts(s) is not None]
aggregator_mols = [Chem.MolFromSmarts(s) for s in AGGREGATOR_SMARTS if Chem.MolFromSmarts(s) is not None]


def has_any(mol, patterns):
    return any(mol.HasSubstructMatch(p) for p in patterns)


print("Annotating structural alerts ...")
post_alert_items = []
alert_counts = defaultdict(int)

for smi, mol in unique_items:
    alerts = {
        "PAINS": pains_catalog.HasMatch(mol),
        "BRENK": brenk_catalog.HasMatch(mol),
        "NIH": nih_catalog.HasMatch(mol),
        "Warhead": has_any(mol, warhead_mols),
        "Aggregator": has_any(mol, aggregator_mols),
    }
    for k, v in alerts.items():
        if v:
            alert_counts[k] += 1

    hard_fail = (
        (FILTER_PAINS and alerts["PAINS"])
        or (FILTER_BRENK and alerts["BRENK"])
        or (FILTER_NIH and alerts["NIH"])
        or (FILTER_WARHEADS and alerts["Warhead"])
        or (FILTER_AGGREGATORS and alerts["Aggregator"])
    )
    if not hard_fail:
        post_alert_items.append((smi, mol, alerts))

funnel["Post hard-alert filter"] = len(post_alert_items)
print(f"  Alert counts: {dict(alert_counts)}")
print(f"  Kept after hard-alert filter: {len(post_alert_items):,}")


# -----------------------------
# Reference novelty / relevance
# -----------------------------
print("Checking reference similarity ...")
ref_std = []
for smi in reference_smiles:
    can, mol = standardize(smi)
    if can is not None:
        ref_std.append((can, mol))
ref_set = {smi for smi, _ in ref_std}
ref_fps = [morgan_fp(mol) for _, mol in ref_std]

novel_items = []
all_max_sims = []
for smi, mol, alerts in post_alert_items:
    fp = morgan_fp(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps) if ref_fps else []
    max_sim = max(sims) if sims else 0.0
    all_max_sims.append(max_sim)
    exact_novel = smi not in ref_set
    similar_enough = max_sim >= SIM_RELEVANCE_LOWER
    not_too_similar = max_sim < SIM_NOVEL_UPPER
    if exact_novel and similar_enough and not_too_similar:
        novel_items.append((smi, mol, max_sim, alerts))

funnel["Novel/relevant"] = len(novel_items)
print(f"  Novel/relevant kept: {len(novel_items):,}")
print(f"  Mean max similarity: {np.mean(all_max_sims):.3f}" if all_max_sims else "  No similarities computed")


# -----------------------------
# Properties and relaxed filters
# -----------------------------
def compute_props(mol):
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    arom_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    fsp3 = rdMolDescriptors.CalcFractionCSP3(mol)
    charge = Chem.GetFormalCharge(mol)
    qed = QED.qed(mol)

    # Aromatic PROPORTION: count aromatic heavy atoms manually
    # (rdMolDescriptors.CalcNumAromaticHeavyAtoms is not available in all RDKit versions)
    n_heavy = mol.GetNumHeavyAtoms()
    if n_heavy:
        n_arom_atoms = sum(
            1 for atom in mol.GetAtoms()
            if atom.GetIsAromatic() and atom.GetAtomicNum() > 1
        )
        arom_proportion = n_arom_atoms / n_heavy
    else:
        n_arom_atoms = 0
        arom_proportion = 0.0

    logs = 0.16 - 0.63 * logp - 0.0062 * mw + 0.066 * rotb - 0.74 * arom_proportion
    return {
        "MW": mw, "LogP": logp, "HBD": hbd, "HBA": hba, "TPSA": tpsa,
        "RotB": rotb, "ArRings": arom_rings, "Fsp3": fsp3, "Charge": charge,
        "logS": logs, "QED": qed,
    }

def relaxed_filter_reason(props):
    ro5_violations = sum([
        props["MW"] > 500,
        props["LogP"] > 5,
        props["HBD"] > 5,
        props["HBA"] > 10,
    ])
    if ro5_violations > MAX_RO5_VIOLATIONS:
        return "Ro5"
    if props["TPSA"] > MAX_TPSA:
        return "TPSA"
    if props["RotB"] > MAX_ROTB:
        return "RotB"
    if props["QED"] < MIN_QED:
        return "QED"
    if props["logS"] < MIN_LOGS:
        return "logS"
    if abs(props["Charge"]) > MAX_ABS_CHARGE:
        return "Charge"
    return "OK"


print("Applying relaxed property filters ...")
property_items = []
fail_reasons = defaultdict(int)
for smi, mol, max_sim, alerts in novel_items:
    props = compute_props(mol)
    reason = relaxed_filter_reason(props)
    if reason == "OK":
        property_items.append((smi, mol, max_sim, alerts, props))
    else:
        fail_reasons[reason] += 1

funnel["Property pass"] = len(property_items)
print(f"  Property pass: {len(property_items):,}")
print(f"  Property fail breakdown: {dict(fail_reasons)}")


# -----------------------------
# Scoring and scaffold annotation (kept for reference/analysis, not for trimming)
# -----------------------------
def get_murcko(mol):
    try:
        framework = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        generic_mol = MurckoScaffold.MakeScaffoldGeneric(Chem.MolFromSmiles(framework)) if framework else None
        generic = Chem.MolToSmiles(generic_mol) if generic_mol is not None else ""
        return framework or "", generic or ""
    except Exception:
        return "", ""


def composite_score(props, max_sim, alerts):
    qed_score = props["QED"]
    sim_score = min(1.0, max_sim / SIM_NOVEL_UPPER) if SIM_NOVEL_UPPER else 0.0
    mw_score = max(0.0, 1.0 - abs(props["MW"] - 380.0) / 280.0)
    logs_score = min(1.0, max(0.0, (props["logS"] + 9.0) / 9.0))
    alert_penalty = 0.03 * sum(bool(v) for v in alerts.values())
    return max(0.0, 0.35 * qed_score + 0.25 * sim_score + 0.20 * mw_score + 0.20 * logs_score - alert_penalty)


print("Scoring and building annotated table ...")
rows = []
for smi, mol, max_sim, alerts, props in property_items:
    framework, generic = get_murcko(mol)
    score = composite_score(props, max_sim, alerts)
    rows.append({
        "SMILES": smi,
        "CompositeScore": score,
        "MaxTanimotoToReference": max_sim,
        "Framework": framework,
        "GenericScaffold": generic,
        **alerts,
        **props,
    })

curated_df = pd.DataFrame(rows)
if curated_df.empty:
    raise RuntimeError("No molecules survived the filters. Check input file / filter thresholds.")

curated_df = curated_df.sort_values("CompositeScore", ascending=False).reset_index(drop=True)
funnel["Curated (kept, all passing)"] = len(curated_df)


# -----------------------------
# Synthetic feasibility annotation only
# -----------------------------
# NOTE: sys.path must be updated BEFORE importing sascorer/scscore, otherwise
# a local sascorer.py / scscore/ folder in the working directory will not be
# found and SA_AVAILABLE will silently stay False.
sys.path.insert(0, os.getcwd())
sys.path.insert(0, str(Path(os.getcwd()) / "scscore"))

try:
    import sascorer
    SA_AVAILABLE = True
except Exception:
    SA_AVAILABLE = False
    print("  SA score unavailable; continuing without it")

try:
    try:
        from scscore.scscore.standalone_model_numpy import SCScorer
    except Exception:
        from scscore.standalone_model_numpy import SCScorer
    _scscorer = SCScorer()
    try:
        _scscorer.restore()
    except TypeError:
        _scscorer.restore(os.path.join("scscore", "models", "full_reaxys_model_1024bool", "model.ckpt-10654.as_numpy.json.gz"))
    SCSCORE_AVAILABLE = True
except Exception as exc:
    SCSCORE_AVAILABLE = False
    print(f"  SCScore unavailable; continuing without it ({exc})")

UNSTABLE_SMARTS = [
    "[N+]#[N-]",
    "[N]=[N]=[N]",
    "C=C=C",
    "C#C#C",
]
unstable_mols = [Chem.MolFromSmarts(s) for s in UNSTABLE_SMARTS if Chem.MolFromSmarts(s) is not None]


def annotate_feasibility(smi):
    """Single-parse feasibility annotation (SA score, SCScore, complexity flags)."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {
            "SA_score": None, "SCScore": None,
            "NRings": None, "NBridgehead": None, "NSpiro": None,
            "UnstableMotif": None, "UndefinedSC": None,
        }

    sa_score = None
    if SA_AVAILABLE:
        try:
            sa_score = float(sascorer.calculateScore(mol))
        except Exception:
            sa_score = None

    sc_score = None
    if SCSCORE_AVAILABLE:
        try:
            _, sc_score = _scscorer.get_score_from_smi(smi)
            sc_score = float(sc_score)
        except Exception:
            sc_score = None

    chiral_centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True)
    return {
        "SA_score": sa_score,
        "SCScore": sc_score,
        "NRings": mol.GetRingInfo().NumRings(),
        "NBridgehead": rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        "NSpiro": rdMolDescriptors.CalcNumSpiroAtoms(mol),
        "UnstableMotif": has_any(mol, unstable_mols),
        "UndefinedSC": sum(1 for _, tag in chiral_centers if tag == "?"),
    }


print("Annotating synthetic feasibility fields ...")
feasibility_rows = [annotate_feasibility(smi) for smi in curated_df["SMILES"]]
feasibility_df = pd.DataFrame(feasibility_rows)
curated_df = pd.concat([curated_df.reset_index(drop=True), feasibility_df], axis=1)
curated_df["SynthWarning"] = (
    (curated_df["SA_score"].fillna(0) > 4.5)
    | (curated_df["UnstableMotif"].fillna(False))
    | (curated_df["UndefinedSC"].fillna(0) > 4)
)

# Cleanest molecules first, but keep everything (no dropping of warning rows).
curated_df = curated_df.sort_values(
    ["SynthWarning", "CompositeScore"], ascending=[True, False]
).reset_index(drop=True)


# -----------------------------
# Export — save everything that passed, no target-count trimming
# -----------------------------
curated_path = f"{OUTPUT_PREFIX}_all_passing.csv"
smiles_path = f"{OUTPUT_PREFIX}_smiles_only.csv"

curated_df.to_csv(curated_path, index=False)
curated_df[["SMILES"]].to_csv(smiles_path, index=False)

print("\nSaved files:")
print(f"  {curated_path} ({len(curated_df):,} molecules)")
print(f"  {smiles_path} ({len(curated_df):,} SMILES)")

print("\nFunnel report:")
previous = None
for stage, count in funnel.items():
    if previous is None:
        print(f"  {stage:<30} {count:>10,}")
    else:
        print(f"  {stage:<30} {count:>10,}  ({count / funnel['Input'] * 100:5.1f}% of input, -{previous - count:,})")
    previous = count

print("\nTop preview:")
display_cols = ["SMILES", "CompositeScore", "MaxTanimotoToReference", "QED", "MW", "LogP", "SA_score", "SynthWarning"]
try:
    display(curated_df[display_cols].head(10))  # noqa: F821 (Jupyter/IPython only)
except NameError:
    print(curated_df[display_cols].head(10).to_string())